In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [3]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name:str)->str:
    """Look up a product by name and return its price, rating , stock and description"""
    p=PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0,
)


agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [4]:
def ask(question:str):
    result = agent.invoke({"messages":[{"role":"user","content":question}]})
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphones.")

[{'type': 'text', 'text': 'The price of the wireless headphones is **$79.99**. \n\n**Product Highlights:**\n* Over-ear Bluetooth design\n* 30-hour battery life\n* Active noise cancellation (ANC)', 'extras': {'signature': 'EoYCCoMCAWkUfRPiC0j3gIpC+9p1C0XmXo33HeV8oxiwZ2NiNnODGoL1ED1e0P3eBhpnNGLF3jT8byrgOJPhxeYKzU34wYhNEzSvUElVVhT/P8XaKhtCpdWxHmAb0khn8ikdr4qwHF3DjosYvs90RvzrQp8tA3kiYWgyiM4oQRfYrkgcToeEs8q85jhlenFoXbnktrNfj33Beor8bbzLH4v9ZIa7w2Dv6QiecnIVM3SSTN9gGsGIpSJiRl6x+SvilsxjrkqHhclpkBF551LMXlfHNEvuyMcJGnrz9fNo7DeYKzlRYE/CMf+2BfBrKjh44uhi7vlhCS2xaCvUOHBI8HgNln1wZjxKtg=='}}]


#### 1. extract product name from question.
#### 2. get_product("wireless headphones")
#### 3. Analyze the answer {"price": 79.99, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."}
#### return human readable answer

In [6]:
REVIEWS = {
    "wireless headphones": {"reviews": 1162, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)


In [7]:
agent2 = create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)
def ask2(question:str):
    result = agent2.invoke({"messages":[{"role":"user","content":question}]})
    print(result["messages"][-1].content)

In [8]:
ask2("how do people like smart watch")

[{'type': 'text', 'text': 'The Smart Watch has a **3.9 out of 5-star rating** based on **340 customer reviews**. \n\nHere are a few quick details about the product:\n* **Price:** $199.99\n* **Key Features:** Heart rate and sleep tracking, 5-day battery life, and water resistance.', 'extras': {'signature': 'EsIDCr8DAWkUfRNTDZwGyDnBeL4DA7fzPxYPBzIU0H+kB7zJPzEK7aD/fDnm114K9aPlynXZ3vpo1iZPR9VbJwf+7Tb34UtRB8XreQobG1EIo71gVzOOFjyOcglSBNJMi+qtzfqXMazDefbJL/kbrdq38UVY+2fYr6rOm5tIPItYWyEUy/dxFLi8O1gPMzBMIfpQPBkbTyUWRFH6osQc8S8msvUhTKyWV7w61/55zbIMrZzj2bPuLamKTEKTruK6dn7eQ8ltdRRtshJxumCTvPqNaXoGcB3wlDkDEN/YZ2hHCtW8G1cEAyg+1liZEcVwgH652Ws3BTo/qfViLoZRyi2oS9GeWbgQxZy5dVA0OobibpJWyN7CM+O9idS9k60UThGEzsGJkSc/1TbfT1dXg0+cn43NveDKf2hEfkmfACDW9fI7jrbHAVGIV1QVMWRihgpNuJIJT8s7tQBFz7xwABF99aUMSxQ1Y8nFzGXPNl3sSIUj1x5TT4hQnGmeBsYHFtLJCmPFFzlS0cbc/Co2m5oVNeuogeD1S2uBxdb4K/20IZ6WOtHUQAsAvbBKKWmD/F+lY95C8ESuXS0pfnjl/Wm0ShPA'}}]
